# Kapitel 19.1 - Einfuehrung in dynamische Webseiten mit CGI und WSGI

Dieses Notebook fuehrt dich Schritt fuer Schritt in die Welt dynamischer Webseiten ein.
Du lernst den Unterschied zwischen statischen und dynamischen Seiten, wie HTTP-Anfragen verarbeitet werden und welche Rolle CGI bzw. WSGI dabei spielen.

Das Ziel ist nicht nur, dass du Code auswendig kennst, sondern dass du den gesamten Ablauf verstehst:
Browser -> Webserver -> Python-Code -> HTML-Antwort.

# Lernziele

Nach diesem Notebook kannst du:

- erklaeren, was eine dynamische Webseite ist
- den Request-Response-Ablauf mit HTTP beschreiben
- CGI und WSGI fachlich sauber voneinander abgrenzen
- die WSGI-Signatur lesen und verstehen
- typische Einsteigerfehler erkennen und vermeiden
- einfache Uebungen ohne Framework selbst loesen

# Voraussetzungen

Du solltest folgende Kapitel bereits kennen:

- Kapitel 07 (Funktionen)
- Kapitel 09 (Module)
- Kapitel 10 (Fehlerbehandlung)
- Kapitel 13 (Textverarbeitung)

Technisch benoetigst du nur Python 3 und Jupyter.
Ein echter Webserver ist fuer die ersten Grundlagen noch nicht notwendig.

# Theorie

## Statisch vs. dynamisch

Eine statische Webseite liefert immer denselben Inhalt aus (z. B. eine feste HTML-Datei).
Eine dynamische Webseite erzeugt Inhalte erst zur Laufzeit, z. B. anhand von Benutzereingaben oder Daten aus einer Datenbank.

Beispiel:
- statisch: 'Willkommen auf unserer Seite'
- dynamisch: 'Willkommen, Fatih' (Name stammt aus Anfrage oder Login)

## Request-Response-Modell

1. Der Browser sendet eine HTTP-Anfrage (Request).
2. Der Server verarbeitet die Anfrage.
3. Python erzeugt eine Antwort (z. B. HTML).
4. Der Browser zeigt das Ergebnis an.

Dieser Zyklus passiert bei jedem Seitenaufruf erneut.

# Erklaerung

## Was ist CGI?

CGI bedeutet Common Gateway Interface.
Ein Webserver startet fuer jede Anfrage ein externes Programm (z. B. ein Python-Skript).
Das Skript liest Umgebungsvariablen und Eingaben, erzeugt Textausgabe und beendet sich wieder.

Vorteil:
- sehr leicht zu verstehen

Nachteil:
- pro Anfrage ein neuer Prozess, dadurch langsam bei hoher Last

## Was ist WSGI?

WSGI bedeutet Web Server Gateway Interface.
Statt fuer jeden Request ein neues Skript zu starten, laeuft eine Python-Anwendung dauerhaft.
Der Webserver uebergibt Request-Daten in standardisierter Form an die Anwendung.

Vorteil:
- deutlich besser skalierbar
- Grundlage vieler Frameworks wie Flask oder Django

# Syntax

## Typische CGI-Ausgabe

```python
print('Content-Type: text/html')
print()
print('<h1>Hallo Welt</h1>')
```

## Typische WSGI-Signatur

```python
def application(environ, start_response):
    status = '200 OK'
    headers = [('Content-Type', 'text/plain; charset=utf-8')]
    start_response(status, headers)
    return [b'Hallo Welt']
```

# Merke

- CGI ist didaktisch sehr gut fuer den Einstieg.
- WSGI ist der industrielle Standard fuer Python-Webanwendungen ohne ASGI.
- Der Browser versteht keine Python-Objekte, sondern nur HTTP-Antworten.
- Eine korrekte Header-Ausgabe ist Pflicht, sonst zeigt der Browser Fehler.

# Parameter

Bei WSGI sind zwei Parameter zentral:

- `environ`: Dictionary-aehnliche Struktur mit Request-Daten
  - z. B. `REQUEST_METHOD`, `PATH_INFO`, `QUERY_STRING`
- `start_response`: Funktion, mit der Status und Header gesetzt werden

Diese Trennung sorgt dafuer, dass die Anwendung klar zwischen Eingabe (Request) und Ausgabe (Response) arbeiten kann.

# Rueckgabewert

Eine WSGI-Anwendung muss ein iterierbares Objekt mit `bytes` zurueckgeben.
In Einsteigerbeispielen ist das meistens eine Liste mit einem Element:

```python
return [b'Hallo']
```

Warum `bytes` und nicht `str`?
HTTP uebertraegt Rohdaten. Text muss daher zuerst nach UTF-8 kodiert werden.

In [ ]:
# Beispiel 1: Simulierter HTTP-Request als Dictionary
request = {
    'method': 'GET',
    'path': '/hallo',
    'query': 'name=Fatih'
}

print('Methode:', request['method'])
print('Pfad   :', request['path'])
print('Query  :', request['query'])

# Beispiel 1 - Erklaerung

Bevor wir echte Serverlogik schreiben, modellieren wir Requests als Dictionary.
Dadurch bleibt der Fokus auf dem Ablauf statt auf Infrastruktur.
Dieses Vorgehen ist didaktisch sehr stark, weil du jeden Schritt gezielt testen kannst.

In [ ]:
# Beispiel 2: Einfache Antwortfunktion wie bei CGI
def baue_html_antwort(titel, text):
    html = f'<html><body><h1>{titel}</h1><p>{text}</p></body></html>'
    return 'Content-Type: text/html\n\n' + html

antwort = baue_html_antwort('Willkommen', 'Dies ist eine dynamische Antwort.')
print(antwort)

# Beispiel 2 - Erklaerung

Die Funktion erzeugt zuerst HTML und setzt davor den HTTP-Header.
Wichtig ist die Leerzeile zwischen Header und Body (`\n\n`).
Ohne diese Trennung kann der Client die Antwort nicht korrekt interpretieren.

In [ ]:
# Beispiel 3: Mini-WSGI-App ohne echten Server (Simulation)
def mini_wsgi_app(environ, start_response):
    name = environ.get('name', 'Unbekannt')
    status = '200 OK'
    headers = [('Content-Type', 'text/plain; charset=utf-8')]
    start_response(status, headers)
    text = f'Hallo {name}, diese Nachricht kommt aus einer WSGI-Funktion.'
    return [text.encode('utf-8')]

def fake_start_response(status, headers):
    print('STATUS :', status)
    print('HEADERS:', headers)

resultat = mini_wsgi_app({'name': 'Kursgruppe'}, fake_start_response)
print('BODY   :', resultat[0].decode('utf-8'))

# Beispiel 3 - Erklaerung

Hier siehst du den Kern von WSGI in minimaler Form:
- Request-Daten werden ueber `environ` gelesen
- Status und Header werden ueber `start_response` gesetzt
- der eigentliche Inhalt wird als `bytes` zurueckgegeben

Dieses Muster taucht spaeter indirekt in Flask oder Django wieder auf.

# Praxisbeispiel

Wir bauen einen sehr kleinen Router, der je nach Pfad unterschiedliche Texte liefert.
So trainierst du bereits den Denkstil moderner Webanwendungen.

In [ ]:
def route(path):
    if path == '/':
        return 'Startseite'
    if path == '/kontakt':
        return 'Kontaktseite'
    if path == '/hilfe':
        return 'Hilfeseite'
    return '404 - Seite nicht gefunden'

for pfad in ['/', '/kontakt', '/hilfe', '/xyz']:
    print(pfad, '->', route(pfad))

# Haeufige Fehler

1. Header vergessen (`Content-Type`).
2. Keine Leerzeile zwischen Header und Body bei CGI.
3. In WSGI `str` statt `bytes` zurueckgeben.
4. Request-Daten nicht absichern (fehlende Keys).
5. Logik und Darstellung unstrukturiert vermischen.

# Best Practice

- Trenne fachliche Logik und HTML-Erzeugung in eigene Funktionen.
- Nutze kleine Hilfsfunktionen fuer Query-Parsing und Response-Aufbau.
- Verwende klare Statuscodes (`200`, `404`, `500`).
- Schreibe frueh kleine Tests fuer Kernfunktionen (Routing, Validierung).
- Behandle Benutzereingaben immer als potenziell fehlerhaft.

# Tipp

Wenn dir Webentwicklung am Anfang komplex vorkommt, reduziere jedes Problem auf drei Fragen:

1. Welche Eingaben kommen rein?
2. Welche Entscheidung trifft mein Code?
3. Welche Antwort geht raus?

Mit diesem Schema kannst du auch groessere Anwendungen sauber aufbauen.

# Uebung

1. Erweitere den Router um die Seite `/impressum`.
2. Schreibe eine Funktion `status_fuer_pfad(path)`, die `200` oder `404` liefert.
3. Erzeuge fuer einen bekannten Pfad eine komplette Textantwort mit Statuszeile und Inhalt.

Arbeite Schritt fuer Schritt und teste jede Funktion einzeln.

In [ ]:
# Loesung
def route(path):
    seiten = {
        '/': 'Startseite',
        '/kontakt': 'Kontaktseite',
        '/hilfe': 'Hilfeseite',
        '/impressum': 'Impressum'
    }
    return seiten.get(path, 'Seite nicht gefunden')

def status_fuer_pfad(path):
    return 200 if path in {'/', '/kontakt', '/hilfe', '/impressum'} else 404

def textantwort(path):
    status = status_fuer_pfad(path)
    inhalt = route(path)
    return f'STATUS: {status}\nINHALT: {inhalt}'

print(textantwort('/impressum'))
print(textantwort('/unbekannt'))

# Zusammenfassung

In diesem ersten Teil hast du das Fundament fuer Kapitel 19 gelegt:

- dynamische Webseiten verstanden
- CGI und WSGI unterschieden
- WSGI-Signatur analysiert
- eigene Mini-Beispiele entwickelt

Im naechsten Notebook gehen wir tief in CGI hinein und arbeiten mit Formulardaten.

# Weiterfuehrende Links

- Python Dokumentation zu CGI (historisch wichtig)
- PEP 3333 (WSGI-Standard)
- Werkzeug und Flask intern auf WSGI-Basis

Hinweis: CGI ist fuer Lernen perfekt, fuer neue Projekte in der Praxis meist WSGI oder ASGI verwenden.

## Technischer Tiefgang

Der Fokus liegt auf reproduzierbaren technischen Entscheidungen statt auf isolierten Einzelbeispielen.
Dabei werden Architektur, Robustheit und Betriebsfaehigkeit gemeinsam betrachtet.

## Zentrale Fachbegriffe

HTTP Semantics
Status Code Family
WSGI Callable
Request Lifecycle
Input Sanitization
Header Validation

In [ ]:
# WSGI-Minibeispiel mit Statuscode
def app(environ, start_response):
    path = environ.get("PATH_INFO", "/")
    if path == "/health":
        start_response("200 OK", [("Content-Type", "text/plain")])
        return [b"ok"]
    start_response("404 Not Found", [("Content-Type", "text/plain")])
    return [b"not found"]

## Fallstudie (Praxis)

Waehle ein realistisches Produktionsszenario und beschreibe systematisch Ursache, Risiko und technische Gegenmassnahmen.
Ergaenze mindestens ein Kriterium fuer Monitoring und ein Kriterium fuer Release-Entscheidungen.

## Haeufige Fehler und Debugging-Checkliste

- Ist das Problem reproduzierbar mit klaren Schritten?
- Sind relevante Signale vorhanden (Logs, Tests, Metriken)?
- Wurde eine konkrete Hypothese getestet und falsifiziert/bestaetigt?
- Ist die Korrektur durch einen Regressionstest abgesichert?
- Wurden Betriebsfolgen und Dokumentation mit aktualisiert?

## Pruefungsfragen und Kurzloesungen

1. Warum ist Reproduzierbarkeit in Fehleranalyse und Betrieb zentral?
Kurzloesung: Ohne reproduzierbare Befunde sind Ursachenanalyse, Fix und Absicherung nicht belastbar.
2. Was unterscheidet technische Begriffe von bloessem Buzzword-Einsatz?
Kurzloesung: Praezise Begriffe steuern messbare Entscheidungen und verbessern Teamkommunikation.
3. Welche Mindestkriterien sollte ein Release-Gate enthalten?
Kurzloesung: Teststatus, Sicherheitschecks, Fehlerbudget und nachvollziehbare Freigabeentscheidung.